# EfficientAD .pt → TensorRT .engine 변환

**실행 환경**: Jetson Orin Nano (JetPack 6.x)

**변환 경로**: PyTorch .pt → ONNX → TensorRT .engine

**핵심**: Teacher + Student + AE 세 네트워크를 단일 ONNX 그래프로 묶어서 내보냄

---
**메모리 절감 예상**

| 포맷 | 크기 | 추론 VRAM | 추론 속도 |
|------|------|-----------|----------|
| .pt (FP32) | ~180MB | ~900MB | baseline |
| .engine FP16 | ~90MB | ~420MB | 2~3× 빠름 |
| .engine INT8 | ~50MB | ~250MB | 3~4× 빠름 |

## Cell 1 — 환경 확인

In [ ]:
import subprocess, sys

# JetPack / TensorRT 버전 확인
print('=== 환경 확인 ===')
for cmd in [
    'python3 -c "import tensorrt; print(\"TensorRT:\", tensorrt.__version__)"',
    'python3 -c "import torch; print(\"PyTorch:\", torch.__version__)"',
    'dpkg -l | grep -i jetpack | head -3',
]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())

import torch
assert torch.cuda.is_available(), 'CUDA 필요'
print(f'\nGPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — 경로 설정

In [ ]:
from pathlib import Path

# ── 여기만 수정 ────────────────────────────────────────────────
PT_PATH     = Path('efficientad_project/models/server_room_efficientad.pt')
ONNX_PATH   = Path('efficientad_project/models/server_room_efficientad.onnx')
ENGINE_PATH = Path('efficientad_project/models/server_room_efficientad.engine')

# FP16 권장 (Orin Nano 기준 정확도 손실 미미, 속도 2배)
# INT8은 캘리브레이션 데이터 필요 → Cell 6에서 선택
PRECISION = 'fp16'   # 'fp16' or 'int8'

# 입력 이미지 크기 (학습 때와 동일)
IMG_SIZE = 256
# ──────────────────────────────────────────────────────────────

assert PT_PATH.exists(), f'파일 없음: {PT_PATH}'
print(f'PT     : {PT_PATH}  ({PT_PATH.stat().st_size/1e6:.1f} MB)')
print(f'ONNX   : {ONNX_PATH}')
print(f'Engine : {ENGINE_PATH}')
print(f'Precision: {PRECISION}')

## Cell 3 — 추론 전용 래퍼 모델 정의

Teacher + Student + AE를 단일 `forward()`로 묶어 ONNX로 내보냄
캘리브레이션 버퍼(t_mean, t_std, q_low, q_high)는 상수로 bake-in

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


# ── 원본 학습 모델 구조 (pt 로드용) ────────────────────────────
class _PDN(nn.Module):
    def __init__(self):
        super().__init__()
        b = models.efficientnet_b4(weights=None)  # 가중치는 .pt에서 로드
        self.features = nn.Sequential(*list(b.features)[:5])
        for p in self.parameters():
            p.requires_grad = False
        self.cuda()
        with torch.no_grad():
            d = torch.zeros(1, 3, 256, 256, device='cuda')
            self.out_ch = self.features(d).shape[1]
    def forward(self, x): return self.features(x)

class _Student(nn.Module):
    def __init__(self, ch):
        super().__init__()
        m = ch // 2
        self.net = nn.Sequential(
            nn.Conv2d(ch, m, 3, padding=1), nn.BatchNorm2d(m), nn.ReLU(True),
            nn.Conv2d(m,  m, 3, padding=1), nn.BatchNorm2d(m), nn.ReLU(True),
            nn.Conv2d(m, ch, 1),
        )
    def forward(self, x): return self.net(x)

class _AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3,   32,  3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(32,  64,  3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(64,  128, 3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.ReLU(True),
        )
        self.btn = nn.Sequential(nn.Conv2d(256,128,1), nn.ReLU(True), nn.Conv2d(128,256,1))
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(256,128,3,stride=2,padding=1,output_padding=1), nn.ReLU(True),
            nn.ConvTranspose2d(128,64, 3,stride=2,padding=1,output_padding=1), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 3,stride=2,padding=1,output_padding=1), nn.ReLU(True),
            nn.ConvTranspose2d(32,  3, 3,stride=2,padding=1,output_padding=1), nn.Sigmoid(),
        )
    def forward(self, x): return self.dec(self.btn(self.enc(x)))

class EfficientAD(nn.Module):
    SZ = 256
    def __init__(self):
        super().__init__()
        self.teacher = _PDN().cuda()
        ch = self.teacher.out_ch
        self.student = _Student(ch).cuda()
        self.ae      = _AE().cuda()
        self.register_buffer('t_mean',    torch.zeros(1, ch, 1, 1, device='cuda'))
        self.register_buffer('t_std',     torch.ones(1,  ch, 1, 1, device='cuda'))
        self.register_buffer('q_low',     torch.tensor(0.0, device='cuda'))
        self.register_buffer('q_high',    torch.tensor(1.0, device='cuda'))
        self.register_buffer('threshold', torch.tensor(0.5, device='cuda'))


# ── ONNX 내보내기용 래퍼 ────────────────────────────────────────
# 캘리브레이션 값을 상수로 bake-in하고
# 입력: (1, 3, 256, 256) 정규화된 이미지
# 출력: (1, 1, 256, 256) 이상 점수맵 (0~1)
class EfficientAD_InferenceWrapper(nn.Module):
    """
    ONNX / TensorRT 변환 전용 래퍼
    - 세 서브네트워크를 단일 그래프로 합침
    - 버퍼값을 상수로 고정 (동적 참조 제거)
    - 출력: 0~1 정규화된 이상 점수맵
    """
    def __init__(self, src: EfficientAD):
        super().__init__()
        self.teacher = src.teacher
        self.student = src.student
        self.ae      = src.ae

        # 버퍼를 상수 텐서로 고정 (ONNX 그래프에 값으로 embed)
        self.register_buffer('t_mean', src.t_mean.clone())
        self.register_buffer('t_std',  src.t_std.clone())
        self.register_buffer('q_low',  src.q_low.clone())
        self.register_buffer('q_high', src.q_high.clone())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (1, 3, 256, 256) CUDA FP16/FP32 — ImageNet 정규화 완료
        returns: (1, 1, 256, 256) 이상 점수맵
        """
        # Teacher (고정)
        t_feat = self.teacher(x)
        t_norm = (t_feat - self.t_mean) / (self.t_std + 1e-8)

        # Student
        s_feat = self.student(t_norm)
        ts_map = ((t_norm - s_feat) ** 2).mean(dim=1, keepdim=True)

        # Autoencoder
        ae_out = self.ae(x)
        ae_map = ((x - ae_out) ** 2).mean(dim=1, keepdim=True)

        # 업샘플 + 결합 + 정규화
        ts_up = F.interpolate(ts_map, size=(256, 256),
                              mode='bilinear', align_corners=False)
        combined = 0.6 * ts_up + 0.4 * ae_map
        combined = (combined - self.q_low) / (self.q_high - self.q_low + 1e-8)
        return combined.clamp(0.0, 1.0)


print('모델 구조 정의 완료')

## Cell 4 — .pt 로드 및 래퍼로 변환

In [ ]:
# 학습된 .pt 로드
ck = torch.load(PT_PATH, map_location='cuda')

src = EfficientAD()
src.load_state_dict(ck['model_state'])
src.t_mean.copy_(ck['teacher_mean'].cuda())
src.t_std.copy_(ck['teacher_std'].cuda())
src.q_low.fill_(ck['q_low'])
src.q_high.fill_(ck['q_high'])
src.eval()

threshold = ck['threshold']   # 나중에 engine 메타에 저장
print(f'threshold (저장) : {threshold:.4f}')

# 래퍼로 변환
wrapper = EfficientAD_InferenceWrapper(src).cuda().eval()

# 동작 확인
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device='cuda')
with torch.no_grad():
    out = wrapper(dummy)
print(f'래퍼 출력 shape : {out.shape}')   # (1, 1, 256, 256)
print(f'래퍼 출력 범위 : {out.min():.3f} ~ {out.max():.3f}')

del src   # 메모리 해제
torch.cuda.empty_cache()

## Cell 5 — ONNX 내보내기

In [ ]:
import torch.onnx

dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device='cuda')

print('ONNX 내보내는 중...')
torch.onnx.export(
    wrapper,
    dummy,
    str(ONNX_PATH),
    export_params=True,
    opset_version=17,          # TRT 10.x 호환 최신 opset
    do_constant_folding=True,  # 상수 표현식 미리 계산 (그래프 최적화)
    input_names=['image'],
    output_names=['anomaly_map'],
    dynamic_axes=None,         # 배치 고정 (Orin Nano 최적화)
)

print(f'ONNX 저장 완료: {ONNX_PATH}  ({ONNX_PATH.stat().st_size/1e6:.1f} MB)')

# ONNX 그래프 간단 검증
try:
    import onnx
    model_onnx = onnx.load(str(ONNX_PATH))
    onnx.checker.check_model(model_onnx)
    print('ONNX 그래프 검증 OK')
except ImportError:
    print('onnx 패키지 없음 — 검증 스킵 (pip install onnx 로 설치 가능)')
except Exception as e:
    print(f'ONNX 검증 경고: {e}')

## Cell 6 — TensorRT 엔진 빌드

**FP16**: 정확도 손실 거의 없음, 속도 2~3× 향상 → 권장

**INT8**: 속도 최대, 정확도 약간 저하, 캘리브레이션 데이터 필요

두 방법 모두 작성되어 있으며 `PRECISION` 값에 따라 자동 선택됩니다.

In [ ]:
import tensorrt as trt
import numpy as np
import cv2
from pathlib import Path
from torchvision import transforms

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)


# ── INT8 캘리브레이터 (PRECISION='int8' 일 때만 사용) ───────────
class EfficientAD_INT8Calibrator(trt.IInt8MinMaxCalibrator):
    """
    INT8 캘리브레이션:
    학습 데이터셋의 정상 이미지 일부를 사용해
    각 레이어의 활성화 범위를 측정 → INT8 양자화 스케일 결정
    """

    def __init__(self, image_dir: str, batch_size: int = 1, n_samples: int = 100):
        super().__init__()
        import pycuda.driver as cuda
        import pycuda.autoinit  # noqa
        self._cuda = cuda

        self._pre = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])

        # 캘리브레이션 이미지 수집
        paths = sorted(Path(image_dir).glob('*.jpg'))[:n_samples]
        assert len(paths) > 0, f'이미지 없음: {image_dir}'
        print(f'INT8 캘리브레이션 이미지: {len(paths)}장')

        self._batches = []
        for p in paths:
            img = cv2.imread(str(p))
            t   = self._pre(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).numpy()
            self._batches.append(t[np.newaxis].astype(np.float32))

        self._idx        = 0
        self._device_inp = cuda.mem_alloc(
            self._batches[0].nbytes
        )
        self._cache_file = str(ENGINE_PATH).replace('.engine', '_int8.cache')

    def get_batch_size(self):
        return 1

    def get_batch(self, names):
        if self._idx >= len(self._batches):
            return None
        self._cuda.memcpy_htod(self._device_inp, self._batches[self._idx])
        self._idx += 1
        return [int(self._device_inp)]

    def read_calibration_cache(self):
        if Path(self._cache_file).exists():
            print('캘리브레이션 캐시 로드')
            return Path(self._cache_file).read_bytes()
        return None

    def write_calibration_cache(self, cache):
        Path(self._cache_file).write_bytes(cache)
        print(f'캘리브레이션 캐시 저장: {self._cache_file}')


# ── TensorRT 엔진 빌드 ──────────────────────────────────────────
def build_engine(onnx_path: str, engine_path: str, precision: str) -> bool:
    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network(
        1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    )
    parser  = trt.OnnxParser(network, TRT_LOGGER)

    with open(onnx_path, 'rb') as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(f'ONNX 파싱 오류: {parser.get_error(i)}')
            return False

    config = builder.create_builder_config()

    # Orin Nano 가용 메모리 기준 workspace 설정 (1GB)
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)

    if precision == 'fp16':
        assert builder.platform_has_fast_fp16, 'FP16 미지원 GPU'
        config.set_flag(trt.BuilderFlag.FP16)
        print('정밀도: FP16')

    elif precision == 'int8':
        assert builder.platform_has_fast_int8, 'INT8 미지원 GPU'
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)  # INT8 fallback용

        # ── INT8 캘리브레이션 이미지 경로 지정 ──────────────────
        CALIB_IMAGE_DIR = 'efficientad_project/dataset/server_room/train'
        calibrator = EfficientAD_INT8Calibrator(
            image_dir=CALIB_IMAGE_DIR,
            n_samples=100,
        )
        config.int8_calibrator = calibrator
        print('정밀도: INT8 + FP16 fallback')

    else:
        print('정밀도: FP32')

    print('엔진 빌드 중... (5~15분 소요, Orin Nano 기준)')
    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        print('엔진 빌드 실패')
        return False

    with open(engine_path, 'wb') as f:
        f.write(serialized)

    size_mb = Path(engine_path).stat().st_size / 1e6
    print(f'엔진 저장 완료: {engine_path}  ({size_mb:.1f} MB)')
    return True


ok = build_engine(str(ONNX_PATH), str(ENGINE_PATH), PRECISION)
print('성공' if ok else '실패')

## Cell 7 — 엔진 검증 (출력값 비교)

In [ ]:
import pycuda.driver as cuda
import pycuda.autoinit  # noqa
import numpy as np
import tensorrt as trt
import cv2
import matplotlib.pyplot as plt


class EngineInferencer:
    """TRT 엔진 추론 헬퍼 (검증 및 노드 교체용)"""

    def __init__(self, engine_path: str):
        logger = trt.Logger(trt.Logger.WARNING)
        with open(engine_path, 'rb') as f:
            self.engine  = trt.Runtime(logger).deserialize_cuda_engine(f.read())
        self.context = self.engine.create_execution_context()
        self.stream  = cuda.Stream()

        self._inp, self._out, self._bindings = None, None, []
        for i in range(self.engine.num_bindings):
            shape = tuple(self.engine.get_binding_shape(i))
            sz    = int(np.prod(shape))
            dt    = trt.nptype(self.engine.get_binding_dtype(i))
            h     = cuda.pagelocked_empty(sz, dt)
            d     = cuda.mem_alloc(h.nbytes)
            self._bindings.append(int(d))
            if self.engine.binding_is_input(i):
                self._inp  = {'h': h, 'd': d, 'shape': shape}
            else:
                self._out  = {'h': h, 'd': d, 'shape': shape}

        from torchvision import transforms
        self._pre = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])

    def infer(self, bgr: np.ndarray) -> np.ndarray:
        """BGR → 이상 점수맵 (H×W float32 0~1)"""
        H, W = bgr.shape[:2]
        inp  = self._pre(
            cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        ).numpy().astype(np.float32)

        np.copyto(self._inp['h'], inp.ravel())
        cuda.memcpy_htod_async(self._inp['d'], self._inp['h'], self.stream)
        self.context.execute_async_v2(self._bindings, self.stream.handle)
        cuda.memcpy_dtoh_async(self._out['h'], self._out['d'], self.stream)
        self.stream.synchronize()

        hm = self._out['h'].reshape(self._out['shape']).squeeze()
        return cv2.resize(hm.copy(), (W, H), interpolation=cv2.INTER_LINEAR)


# 엔진 로드
engine_inf = EngineInferencer(str(ENGINE_PATH))
print('엔진 로드 완료')

# 테스트 이미지로 PT vs Engine 출력 비교
test_imgs = sorted(Path('efficientad_project/dataset/server_room/test/good').glob('*.jpg'))[:3]
if test_imgs:
    fig, axes = plt.subplots(len(test_imgs), 3, figsize=(15, 4*len(test_imgs)))

    for row, tp in enumerate(test_imgs):
        frame = cv2.imread(str(tp))

        # PT 추론
        import torch
        from torchvision import transforms as T
        pre = T.Compose([
            T.ToPILImage(), T.Resize((IMG_SIZE,IMG_SIZE)),
            T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        x = pre(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).unsqueeze(0).cuda()
        with torch.no_grad():
            hm_pt = wrapper(x).squeeze().cpu().numpy()

        # Engine 추론
        hm_trt = engine_inf.infer(frame)
        hm_trt_resized = cv2.resize(hm_trt, (IMG_SIZE, IMG_SIZE))

        # 차이
        diff = np.abs(hm_pt - hm_trt_resized)
        max_diff = diff.max()

        axes[row][0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axes[row][0].set_title('Original')
        axes[row][1].imshow(hm_pt,  cmap='jet', vmin=0, vmax=1)
        axes[row][1].set_title(f'PyTorch PT  max={hm_pt.max():.3f}')
        axes[row][2].imshow(hm_trt_resized, cmap='jet', vmin=0, vmax=1)
        axes[row][2].set_title(f'TRT Engine  max={hm_trt_resized.max():.3f}  diff={max_diff:.4f}')
        for ax in axes[row]: ax.axis('off')

    # FP16 기준 허용 오차: 0.01 이하면 정상
    plt.suptitle('PT vs TRT Engine comparison\n'
                 'diff < 0.01: OK  |  diff > 0.05: precision issue', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('테스트 이미지 없음 — 경로 확인 필요')

## Cell 8 — 속도 벤치마크

In [ ]:
import time
import numpy as np

dummy_frame = np.random.randint(0, 255, (1080, 1920, 3), dtype=np.uint8)
N = 50

# PT 속도
from torchvision import transforms as T
pre = T.Compose([
    T.ToPILImage(), T.Resize((IMG_SIZE,IMG_SIZE)),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

for _ in range(5):  # 워밍업
    x = pre(cv2.cvtColor(dummy_frame, cv2.COLOR_BGR2RGB)).unsqueeze(0).cuda()
    with torch.no_grad(): wrapper(x)
torch.cuda.synchronize()

t0 = time.perf_counter()
for _ in range(N):
    x = pre(cv2.cvtColor(dummy_frame, cv2.COLOR_BGR2RGB)).unsqueeze(0).cuda()
    with torch.no_grad(): wrapper(x)
torch.cuda.synchronize()
pt_ms = (time.perf_counter() - t0) / N * 1000

# TRT 속도
for _ in range(5):  # 워밍업
    engine_inf.infer(dummy_frame)

t0 = time.perf_counter()
for _ in range(N):
    engine_inf.infer(dummy_frame)
trt_ms = (time.perf_counter() - t0) / N * 1000

print('=== 속도 벤치마크 (1080p 입력) ===')
print(f'PyTorch .pt  : {pt_ms:6.1f} ms/frame  ({1000/pt_ms:5.1f} FPS)')
print(f'TRT {PRECISION.upper():5s}   : {trt_ms:6.1f} ms/frame  ({1000/trt_ms:5.1f} FPS)')
print(f'속도 향상     : {pt_ms/trt_ms:.1f}×')

# VRAM 사용량
torch.cuda.empty_cache()
before = torch.cuda.memory_allocated()
_ = engine_inf.infer(dummy_frame)
print(f'\nTRT 추론 VRAM: {torch.cuda.memory_allocated()/1e6:.0f} MB')

## Cell 9 — 엔진 메타데이터 저장 및 노드 교체용 클래스 출력

In [ ]:
import json

# 엔진과 함께 저장할 메타데이터 (threshold 등)
meta_path = ENGINE_PATH.with_suffix('.json')
meta = {
    'zone':      ENGINE_PATH.stem.replace('_efficientad', ''),
    'threshold': threshold,
    'precision': PRECISION,
    'img_size':  IMG_SIZE,
    'engine':    ENGINE_PATH.name,
}
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'메타데이터 저장: {meta_path}')
print(json.dumps(meta, indent=2))

print()
print('=== 변환 완료 ===')
print(f'  ONNX   : {ONNX_PATH}  ({ONNX_PATH.stat().st_size/1e6:.1f} MB)')
print(f'  Engine : {ENGINE_PATH}  ({ENGINE_PATH.stat().st_size/1e6:.1f} MB)')
print(f'  Meta   : {meta_path}')
print()
print('obstacle_detector_node.py 에서 EfficientAD 클래스를')
print('아래 TRTEfficientAD 클래스로 교체하면 됩니다.')
print('(load_efficientad 함수도 load_efficientad_trt로 교체)')

## Cell 10 — 노드 교체용 TRT 클래스

이 셀의 코드를 `obstacle_detector_node.py`의 `EfficientAD` 클래스와
`load_efficientad` 함수 대신 사용하세요.

In [ ]:
# ── obstacle_detector_node.py 교체용 코드 ──────────────────────
# EfficientAD 클래스와 load_efficientad 를 아래로 교체

REPLACEMENT_CODE = '''
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # noqa


class TRTEfficientAD:
    """
    TensorRT 엔진 기반 EfficientAD 추론 클래스
    obstacle_detector_node.py의 EfficientAD 클래스를 1:1 대체
    .infer(), .threshold 인터페이스 동일 유지
    """

    def __init__(self, engine_path: str, meta_path: str):
        import json
        with open(meta_path) as f:
            meta = json.load(f)

        self._threshold = meta['threshold']
        self._sz        = meta['img_size']        # 256

        logger = trt.Logger(trt.Logger.WARNING)
        with open(engine_path, 'rb') as f:
            self._engine  = trt.Runtime(logger).deserialize_cuda_engine(f.read())
        self._context = self._engine.create_execution_context()
        self._stream  = cuda.Stream()

        self._inp, self._out, self._bindings = None, None, []
        for i in range(self._engine.num_bindings):
            shape = tuple(self._engine.get_binding_shape(i))
            sz    = int(np.prod(shape))
            dt    = trt.nptype(self._engine.get_binding_dtype(i))
            h     = cuda.pagelocked_empty(sz, dt)
            d     = cuda.mem_alloc(h.nbytes)
            self._bindings.append(int(d))
            if self._engine.binding_is_input(i):
                self._inp = {'h': h, 'd': d}
            else:
                self._out = {'h': h, 'd': d, 'shape': shape}

        from torchvision import transforms
        self._pre = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((self._sz, self._sz)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                  [0.229, 0.224, 0.225]),
        ])

    @property
    def threshold(self):
        """obstacle_detector_node.py와 인터페이스 동일"""
        class _T:
            def __init__(s, v): s.v = v
            def cpu(s): return s
            def __float__(s): return s.v
        return _T(self._threshold)

    def infer(self, bgr: np.ndarray) -> tuple:
        """→ (heatmap H×W float32 0~1, max_score float)"""
        H, W = bgr.shape[:2]
        inp  = self._pre(
            cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        ).numpy().astype(np.float32)

        np.copyto(self._inp['h'], inp.ravel())
        cuda.memcpy_htod_async(self._inp['d'], self._inp['h'], self._stream)
        self._context.execute_async_v2(self._bindings, self._stream.handle)
        cuda.memcpy_dtoh_async(self._out['h'], self._out['d'], self._stream)
        self._stream.synchronize()

        hm = self._out['h'].reshape(self._out['shape']).squeeze().copy()
        hm = cv2.resize(hm, (W, H), interpolation=cv2.INTER_LINEAR)
        return hm, float(hm.max())


def load_efficientad_trt(engine_path: str) -> TRTEfficientAD:
    """obstacle_detector_node.py의 load_efficientad 대체"""
    meta_path = engine_path.replace('.engine', '.json')
    return TRTEfficientAD(engine_path, meta_path)
'''

print(REPLACEMENT_CODE)